**Ingest Races - Incremental**

Reads `races.csv` from the batch landing folder, adds metadata, and writes to `formula1_incr.bronze.races` partitioned by `batch_id`.

**Load config and helpers**

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze_helpers

In [0]:
dbutils.widgets.text('p_batch_id','')
v_batch_id= dbutils.widgets.get('p_batch_id')

In [0]:
v_batch_id

In [0]:
import pyspark.sql.functions as f

In [0]:
source_file = f'{loding_folder_path}/{v_batch_id}/races.csv' # to replace the load data in read api
table_name = f'{catalog_name}.{bronze_schema}.races' # to replace the save table in write api 

In [0]:
display(source_file)

**Define schema and read CSV**

In [0]:
from pyspark.sql.types import *
races_schema = StructType([
  StructField('season', IntegerType(), True),
  StructField('round', IntegerType(), True),
  StructField('url', StringType(), True),
  StructField('raceName',StringType(), True),
   StructField('date', DateType(), True),
  StructField('circuitId', StringType(), True)
 ])

In [0]:
races_df = (
  spark.read
  .format('csv')
  .option('header', True)
  .schema(races_schema)
  .load(source_file))
display(races_df)

**Add metadata columns**

In [0]:
races_final_df = add_ingestion_metadata(races_df)
display(races_final_df)

**Write to bronze Delta table** (overwrite per batch partition)

In [0]:
write_to_bronze(
    input_df=races_final_df,
    table_name=table_name,
    batch_id=v_batch_id
)

In [0]:
display(spark.table(table_name))